# Atividade Prática 4


Aluno: João Gabriel Angelo Bradachi

Professor: Fabrício Silva

Objetivo: Aplicar técnicas engenharia de atributos no texto da mensagem para gerar atributos relevantes


In [1]:
%pip install nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 4.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.6/804.6 kB 4.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 3.4 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder 
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

In [ ]:
df_spam = pd.read_csv("spam-dataset.csv")
df_spam

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [ ]:
# Tokenização
# remoção de stop words 
# Lematização e Radicalização -> pode ser que piora, vamos comparar depois

# Vamos aplicar em:

# TF-IDF e rodar em XGB

# similaridade de cossenos e ver se o que chegou é mais similar aos que são spam do que com os que não são spam

# Word2Vector

stop_words = set(stopwords.words('english'))

# Filtrar as palavras que não estão na lista de stopwords


# Sem Lematização e Radicalização
def remove_sw(texto):
    tokens = word_tokenize(texto.lower())
    filtered_tokens = [word for word in tokens if word not in stop_words]
    clean_text = " ".join(filtered_tokens)
    return clean_text

df_spam['sem_sw'] = df_spam['Message'].apply(remove_sw)
df_spam

,Category,Message,sem_sw
0,ham,"Go until jurong point, crazy.. Available only ...","go jurong point , crazy .. available bugis n g..."
1,ham,Ok lar... Joking wif u oni...,ok lar ... joking wif u oni ...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,ham,U dun say so early hor... U c already then say...,u dun say early hor ... u c already say ...
4,ham,"Nah I don't think he goes to usf, he lives aro...","nah n't think goes usf , lives around though"
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,2nd time tried 2 contact u. u £750 pound prize...
5568,ham,Will ü b going to esplanade fr home?,ü b going esplanade fr home ?
5569,ham,"Pity, * was in mood for that. So...any other s...","pity , * mood . ... suggestions ?"
5570,ham,The guy did some bitching but I acted like i'd...,guy bitching acted like 'd interested buying s...


In [ ]:
# Aplicando TF-IDF

# 1. Defina o seu corpus (conjunto de documentos/textos)
documentos = [
    "O aprendizado de máquina é fantástico.",
    "Processamento de linguagem natural e aprendizado de máquina.",
    "O TF-IDF é uma técnica para processamento de texto."
]

# 2. Inicialize o construtor do TF-IDF
# O parâmetro stop_words=None pode ser alterado para remover palavras irrelevantes
vectorizer = TfidfVectorizer()

# 3. Aplique o algoritmo e transforme os textos em uma matriz de frequências
tfidf_matrix = vectorizer.fit_transform(documentos)

# 4. Extraia as palavras (colunas da matriz)
nomes_recursos = vectorizer.get_feature_names_out()

# 5. Transforme em um DataFrame para visualizar facilmente como uma tabela
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=nomes_recursos)

# Exibe o resultado
print(df_tfidf)

In [ ]:
# Testando no XGB 